# 📅 Day 15 — Mock Interview Practice

**Week 3 · Day 1 (Week 3 begins) · Level: ADVANCED (stepping into HARD)**

Week 3 opens: **product-company level** questions — complex end-to-end scenarios, optimization trade-offs, and system-design aspects. 18 questions across 8 categories — 16 [ADVANCED] + 2 [HARD]. Week 3 is where answers stop being *correct* and start being *judgment*: expect to defend trade-offs out loud.

| # | Category | Questions | Level |
| --- | --- | --- | --- |
| 1 | SQL Advanced | Q1-Q2 | ADVANCED |
| 2 | Python — Loop/Function/List/Dict | Q3-Q4 | ADVANCED |
| 3 | PySpark Coding | Q5-Q6 | ADVANCED |
| 4 | Spark Optimization & Deep Spark | Q7-Q8 | ADVANCED |
| 5 | ETL — Failures, Late Data, Schema, Fault Tolerance | Q9-Q11 | ADVANCED → HARD |
| 6 | Data Modeling | Q12-Q13 | ADVANCED |
| 7 | Snowflake | Q14-Q15 | ADVANCED |
| 8 | Monitoring, Logging & Orchestration | Q16-Q18 | ADVANCED → HARD |

**Today's thread**: baseline-aware anomaly detection (Q1) → range joins (Q2) → prefix sums (Q3) — the same "sliding baseline" theme three ways. Q11 enters 2025-2026 trending territory: **RAG / GenAI data pipelines**.

> Days 22+ (Week 4): FAANG / top product-company level — Google, Amazon, Meta, Uber, Netflix — streaming, Lakehouse, and GenAI pipeline topics included.

---

## Q1 [ADVANCED] SQL Advanced — Trailing-Window Anomaly Detection (Per-Merchant z-Score Band)

**Scenario**: A fintech fraud alerting system must flag merchant days where spend deviates from **that merchant's own recent history** — not from a global average (a $300 day is normal for a supermarket, terrifying for a coffee cart). You get one pass over the data.

**Requirements**:
1. For each merchant-day, compute the **trailing 7-day mean and sample stddev** — with the **current day EXCLUDED** from the baseline (else a spike inflates its own threshold and hides itself).
2. **Cold start**: only flag once at least 5 prior observations exist.
3. Flag `SPIKE` when amount > avg + 3·σ and `DROP` when amount < avg − 3·σ; also output the z-score.
4. Explain: why `ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING` (not `CURRENT ROW`)? Why sample stddev (`STDDEV_SAMP`)? What changes with `RANGE` on a date if days are missing?

**Sample data & expected**: coffee is ~$100/day with a one-day $300 spike (day 9) → SPIKE; bagels is ~$50/day with a $12 day (day 10) → DROP. Nothing else flags.

In [0]:
# ── Solution Q1: three window stats in one pass, baseline excludes today ────
spend_rows = [
    ("coffee", 1, 100), ("coffee", 2, 102), ("coffee", 3, 98), ("coffee", 4, 101),
    ("coffee", 5, 99), ("coffee", 6, 100), ("coffee", 7, 103), ("coffee", 8, 97),
    ("coffee", 9, 300), ("coffee", 10, 100),
    ("bagels", 1, 50), ("bagels", 2, 51), ("bagels", 3, 49), ("bagels", 4, 52),
    ("bagels", 5, 50), ("bagels", 6, 48), ("bagels", 7, 51), ("bagels", 8, 49),
    ("bagels", 9, 50), ("bagels", 10, 12),
]
(spark.createDataFrame(spend_rows, ["merchant", "dt", "amount_usd"])
 .createOrReplaceTempView("merchant_spend"))

anomalies = spark.sql("""
WITH stats AS (
  SELECT merchant, dt, amount_usd,
         AVG(amount_usd)        OVER w AS prev_avg,
         STDDEV_SAMP(amount_usd) OVER w AS prev_std,
         COUNT(*)               OVER w AS prev_n
  FROM merchant_spend
  WINDOW w AS (PARTITION BY merchant ORDER BY dt
               ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING)   -- 7 prior days, NO current row
)
SELECT merchant, dt, amount_usd,
       ROUND(prev_avg, 2)  AS trailing_avg_7d,
       ROUND(prev_std, 2)  AS trailing_std_7d,
       ROUND((amount_usd - prev_avg) / NULLIF(prev_std, 0), 2) AS z_score,
       CASE
         WHEN prev_n >= 5 AND amount_usd > prev_avg + 3 * prev_std THEN 'SPIKE'
         WHEN prev_n >= 5 AND amount_usd < prev_avg - 3 * prev_std THEN 'DROP'
         ELSE 'OK'
       END AS anomaly_flag
FROM stats
ORDER BY merchant, dt
""")

cf = {r["dt"]: r["anomaly_flag"] for r in anomalies.filter("merchant = 'coffee'").collect()}
bf = {r["dt"]: r["anomaly_flag"] for r in anomalies.filter("merchant = 'bagels'").collect()}
assert cf[9] == "SPIKE" and cf[10] == "OK" and bf[10] == "DROP" and bf[9] == "OK"
anomalies.filter("anomaly_flag != 'OK'").show()

print("Talk track — the four gotchas:")
print("  1) Baseline EXCLUDES the current row: with CURRENT ROW inside, a 3x spike lifts")
print("     the mean and inflates sigma so much that z shrinks below 3 — the anomaly hides itself.")
print("  2) STDDEV_SAMP (n-1): with 7 points, sample stddev is the honest estimator; and")
print("     NULLIF(prev_std, 0) guards a flat window (sigma 0) from dividing by zero.")
print("  3) ROWS counts rows; RANGE on a date value would give 'prior 7 CALENDAR days' —")
print("     if a merchant skips weekends, ROWS still uses the last 7 observations. Pick deliberately.")
print("  4) Cold start (prev_n >= 5): without it, day 2 of a new merchant has a 1-point")
print("     baseline and pages on-call for nothing — the classic false-positive flood.")

## Q2 [ADVANCED] SQL Advanced — Range Joins: Finding Double-Booked Meeting Rooms

**Scenario**: An office-booking table stores half-open intervals `[start_ts, end_ts)`. Find every pair of bookings for the **same room** that conflict, and by how many minutes — the input to a billing dispute and to a UX bug fix.

**Requirements**:
1. Write the **interval-overlap predicate** for half-open intervals and explain why it is NOT `a.start < b.start AND a.end > b.end`-style thinking.
2. Emit each conflicting pair exactly once (no self-pairs, no mirrored duplicates) with `overlap_minutes`.
3. Explain the **range join problem**: why no equi-join can express this, what the naive plan degrades to, and how the `RANGE_JOIN` hint with a bin size helps Spark/Databricks build a partition-band index.
4. **Scale follow-up**: 10M bookings, 200k rooms — what is the bin size trade-off?

**Sample data & expected**: roomA has 4 bookings (B1 9–10, B2 10–11, B3 9:30–10:30, B4 13–14) and roomB one long one. Expected conflicts: B1×B3 and B2×B3, 30 minutes each (B2 follows B1 back-to-back — **not** a conflict).

In [0]:
# ── Solution Q2: half-open overlap predicate + RANGE_JOIN hint ──────────────
bookings_rows = [
    ("B1", "roomA", "2026-10-01 09:00:00", "2026-10-01 10:00:00"),
    ("B2", "roomA", "2026-10-01 10:00:00", "2026-10-01 11:00:00"),
    ("B3", "roomA", "2026-10-01 09:30:00", "2026-10-01 10:30:00"),   # the double-book
    ("B4", "roomA", "2026-10-01 13:00:00", "2026-10-01 14:00:00"),
    ("B5", "roomB", "2026-10-01 09:00:00", "2026-10-01 12:00:00"),
]
bk = (spark.createDataFrame(bookings_rows, ["booking_id", "room", "start_ts", "end_ts"])
      .withColumn("start_ts", F.to_timestamp("start_ts"))
      .withColumn("end_ts", F.to_timestamp("end_ts")))
bk.createOrReplaceTempView("bookings")

overlaps = spark.sql("""
SELECT /*+ RANGE_JOIN(a, 16) */
       a.booking_id AS booking_a, b.booking_id AS booking_b, a.room,
       ROUND((UNIX_TIMESTAMP(LEAST(a.end_ts, b.end_ts)) -
              UNIX_TIMESTAMP(GREATEST(a.start_ts, b.start_ts))) / 60.0, 1) AS overlap_minutes
FROM bookings a
JOIN bookings b
  ON  a.room = b.room                 -- equality part prunes first
  AND a.booking_id < b.booking_id      -- canonical order: each pair emitted once, no self-join
  AND a.start_ts < b.end_ts           -- half-open overlap predicate:
  AND b.start_ts < a.end_ts           -- [a) and [b) intersect  <==>  a.s < b.e AND b.s < a.e
ORDER BY booking_a, booking_b
""")

ov = {(r["booking_a"], r["booking_b"]): float(r["overlap_minutes"]) for r in overlaps.collect()}
assert ov == {("B1", "B3"): 30.0, ("B2", "B3"): 30.0}, ov   # B1/B2 back-to-back: NOT a conflict
overlaps.show()

print("Talk track:")
print("  The overlap predicate: two half-open intervals intersect iff each starts before the")
print("  other ends. Touching endpoints (10:00 = 10:00) do NOT conflict — that's the point of half-open.")
print("  Why range joins are special: the join key is an INEQUALITY, so hash/equi-join machinery")
print("  cannot apply; naive plan = partition by the equality keys then nested-loop within them.")
print("  RANGE_JOIN(a, 16) tells the planner to bucket BOTH sides into 16-minute bands keyed by")

print("  band(floor(ts/16min)) so candidate pairs meet in a hash join on (room, band) —")
print("  closer to O(n log n) than O(n^2). Bin size = expected overlap width: too small -> one")
print("  booking replicates across many bands (write amplification); too large -> fat buckets,")
print("  quadratic again. Set it near the 99th-percentile booking duration.")
print("  Interview bonus: same machinery answers sessionization, con-flict detection in")
print("  scheduling systems, and ASOF/temporal joins ('which price was active at order time').")

## Q3 [ADVANCED] Python — Subarray Sum Equals K (Prefix Sums + Hash Map)

**Scenario**: A payments stream is summarized as a list of signed transaction deltas. Accounting wants every **contiguous window whose net is exactly K** — e.g. find all days ranges netting to $7K. Negative values are common (refunds), so sliding-window tricks from all-positive problems do NOT apply.

**Requirements**:
1. Count all contiguous subarrays summing to `k` in **O(n)** — negatives must be handled.
2. Also return the index range `(i, j)` of the **first** such subarray, or `None`.
3. Explain the core insight: prefix sums + what the hash map stores, and why the all-positive sliding window fails here.
4. State the follow-up variants interviewers ask: max-size subarray, subarrays divisible by K (modulo trick), 2D matrix variant.

**Tests**: `[3,4,7,2,-3,1,4,2], k=7` → count 4; `[1,2,3], k=3` → 2; `[1,-1,0], k=0` → 3; `[5], k=3` → (0, None).

In [0]:
# ── Solution Q3: prefix-sum + hashmap, single pass ──────────────────────────
from collections import defaultdict

def subarrays_sum_k(nums, k):
    """Returns (count, first_index_range_or_None)."""
    first_idx = {0: -1}                       # prefix value -> first index where seen
    freq = defaultdict(int); freq[0] = 1      # prefix value -> occurrences so far
    running, count, example = 0, 0, None
    for i, x in enumerate(nums):
        running += x                          # prefix sum up to i
        need = running - k                    # a subarray (j+1..i) sums to k
        if need in freq:                      #   iff some earlier prefix == running - k
            count += freq[need]
            if example is None and need in first_idx:
                example = (first_idx[need] + 1, i)
        if running not in first_idx:
            first_idx[running] = i
        freq[running] += 1
    return count, example

assert subarrays_sum_k([3, 4, 7, 2, -3, 1, 4, 2], 7) == (4, (0, 2))
assert subarrays_sum_k([1, 2, 3], 3) == (2, (0, 1))
assert subarrays_sum_k([1, -1, 0], 0) == (3, (0, 1))
assert subarrays_sum_k([5], 3) == (0, None)
print("tests pass:", subarrays_sum_k([3, 4, 7, 2, -3, 1, 4, 2], 7))

print("\nTalk track:")
print("  Core identity: sum(i..j) = prefix(j) - prefix(i-1). So sum(i..j) = k  <==>")
print("  prefix(j) - k == some earlier prefix. One pass, hashing every prefix seen: O(n) time, O(n) space.")
print("  Why sliding window fails: it relies on monotonicity (adding a positive always grows")
print("  the sum) to shrink/expand safely. With negatives, growing the window can DECREASE the")
print("  sum, so the two-pointer invariant collapses.")
print("  Variants to pre-think: (1) LONGEST subarray = k: store FIRST index of each prefix only;")
print("  (2) count divisible by k: hash prefix % k with math-mod for negatives;")
print("  (3) 2D matrix = k: prefix sums per row pairs — O(n^2) over column pairs.")

## Q4 [ADVANCED] Python — Trie Autocomplete Engine

**Scenario**: Build the lookup core of a search box: as the user types `app`, suggest the 3 most popular completions in real time.

**Requirements**:
1. Implement a **Trie** (prefix tree) with `insert(word, freq)` and `top_k(prefix, k)` returning the k highest-frequency words with the given prefix (ties broken alphabetically).
2. State the time complexity of insert, prefix navigation, and the completion scan for a trie with n words of average length L.
3. **Follow-up they will ask**: how to make `top_k` O(prefix length) instead of a subtree walk — describe the precomputation (no need to implement).
4. Pass the tests below.

**Tests**: insert (apple,5), (app,3), (application,7), (apply,2), (ape,1), (banana,9) → `top_k("app", 3) == ["application", "apple", "app"]`, `top_k("appl") == ["apple", "apply"]`, `top_k("zzz") == []`.

In [0]:
# ── Solution Q4: trie with frequency-ranked autocomplete ─────────────────────
class TrieNode:
    __slots__ = ("children", "is_word", "freq")   # __slots__: leaner nodes, faster GC
    def __init__(self):
        self.children, self.is_word, self.freq = {}, False, 0

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word, freq):                 # O(len(word))
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
        node.is_word, node.freq = True, freq

    def _walk(self, node, prefix, out):           # subtree DFS
        if node.is_word:
            out.append((prefix, node.freq))
        for ch, child in sorted(node.children.items()):
            self._walk(child, prefix + ch, out)

    def top_k(self, prefix, k=3):                 # O(len(prefix)) + subtree walk
        node = self.root
        for ch in prefix:                          # navigate to the prefix node
            if ch not in node.children:
                return []
            node = node.children[ch]
        words = []
        self._walk(node, prefix, words)
        return [w for w, _ in sorted(words, key=lambda x: (-x[1], x[0]))[:k]]

t = Trie()
for w, f in [("apple", 5), ("app", 3), ("application", 7), ("apply", 2), ("ape", 1), ("banana", 9)]:
    t.insert(w, f)
assert t.top_k("app", 3) == ["application", "apple", "app"]
assert t.top_k("appl") == ["apple", "apply"]
assert t.top_k("zzz") == []
print(t.top_k("app", 3), "|", t.top_k("appl"))

print("\nTalk track:")
print("  Complexity: insert O(L); prefix nav O(L); completion = O(subtree size) — fine for")
print("  lexicons up to ~100k, plus a frequency sort of the matches.")
print("  O(L) top_k precomputation: store on EVERY node the top-k (word, freq) list of its")
print("  subtree, updated on insert (merge the child lists) — classic interview bonus; cost")
print("  is k extra entries per node of memory.")
print("  Real systems (Elasticsearch, Redis) rank by score f(freq, recency, typing distance)")
print("  and usually post-filter with an n-gram/FST index — mention it and you sound senior.")

## Q5 [ADVANCED] PySpark Coding — Mutual Friends (Graph Problem on a Social Network)

**Scenario**: You have a friendships edge table (symmetric). Product asks, per pair of friends, **how many friends they have in common** — the feature behind "people you may know".

**Requirements**:
1. Given edges (u, v) stored one-direction per row (mirror them yourself), compute for every friend pair (a < b) the set of mutual friends.
2. Do it with joins/aggregations only — no UDFs, no collect to driver.
3. Explain the pattern: what does joining the edge table with itself on the common neighbor compute, and why must you then filter to actual friend pairs?
4. Note the scale risk: which real-world node type explodes this computation, and which earlier technique handles it?

**Sample edges**: (1,2), (1,3), (2,3), (2,4), (3,4), (3,5), (1,5). Expected: pair (1,3) → mutual [2, 5]; pair (2,3) → [1, 4]; pair (1,2) → [3].

In [0]:
# ── Solution Q5: mutual friends via self-join on the common neighbor ────────
raw_edges = [(1, 2), (1, 3), (2, 3), (2, 4), (3, 4), (3, 5), (1, 5)]
sym = raw_edges + [(v, u) for u, v in raw_edges]        # mirror to both directions
e = spark.createDataFrame(sym, ["user", "friend"])

# f1(a, x): x is a friend of a      f2(b, x): x is a friend of b
# joining on x enumerates every COMMON NEIGHBOR of the pair (a, b)
f1 = e.select(F.col("user").alias("a"), F.col("friend").alias("x"))
f2 = e.select(F.col("user").alias("b"), F.col("friend").alias("x"))

pairs = (e.select(F.least("user", "friend").alias("pa"),
                  F.greatest("user", "friend").alias("pb"))).distinct()

mutual = (f1.join(f2, "x")                              # all (a, b, x) sharing neighbor x
            .filter((F.col("a") < F.col("b")) &        # canonical pair order, no self-pairs
                    (F.col("x") != F.col("a")) & (F.col("x") != F.col("b")))
            .join(pairs, (F.col("a") == F.col("pa")) &  # keep only REAL friend pairs
                           (F.col("b") == F.col("pb")))
            .groupBy("pa", "pb")
            .agg(F.collect_set("x").alias("mutual_friends"))
            .orderBy("pa", "pb"))

got = {(r["pa"], r["pb"]): sorted(r["mutual_friends"]) for r in mutual.collect()}
assert got[(1, 3)] == [2, 5] and got[(2, 3)] == [1, 4] and got[(1, 2)] == [3], got
mutual.show(truncate=False)

print("Talk track:")
print("  The self-join on the COMMON NEIGHBOR x produces candidate pairs (a, b, x) where x")
print("  is friends with both — but that does NOT mean a and b are friends with EACH OTHER")
print("  (that's 'friends-of-friends'). The second join against the pairs relation enforces")
print("  that (a, b) is itself an edge. Filtering x != a, x != b removes the pair members")
print("  themselves from the mutual set.")
print("  Scale risk: celebrities. A node with 10M friends squares into 10^14 candidate pairs")
print("  for its neighborhood — cap degree, sample, or two-step-bucket (and the skew/salting")
print("  + AQE skew-split techniques from earlier days apply to this join).")

## Q6 [ADVANCED] PySpark Coding — Nightly Batch Model Scoring with `mapInPandas`

**Scenario**: Risk scoring runs nightly over 10M customers. The model is a **pandas-based** scoring function (lib only exists in pandas-land). A previous attempt used `df.toPandas()` on the driver and OOM'd; another used a row-at-a-time Python UDF and took 6 hours.

**Requirements**:
1. Explain why `toPandas()`/`collect()` is the wrong shape for 10M rows, and why row-UDFs are slow (serialization + Python GIL).
2. Score the population with **`mapInPandas`**: Arrow-serialized batches in, batches out, no driver materialization. Score = 0.5·f1 + 0.3·f2 + 0.2·f3, band = LOW ≤ 40 < MED ≤ 70 < HIGH.
3. State the contract your function must honor (schema in/out, no global mutable state, deterministic) and how `applyInPandas` differs (grouping contract).
4. Note the memory bound: what happens if one batch is too big, and how to control batch size.

**Expected output**: scored table for 8 customers + the talk-track.

In [0]:
# ── Solution Q6: Arrow-batched scoring without touching the driver ─────────────
def score_batches(itr):
    """Generator: consumes pandas batches, yields pandas batches. Runs on WORKERS."""
    import pandas as pd
    for pdf in itr:                       # Arrow batch in
        pdf = pdf.copy()
        pdf["score"] = 0.5 * pdf["f1"] + 0.3 * pdf["f2"] + 0.2 * pdf["f3"]
        pdf["risk_band"] = pd.cut(pdf["score"], bins=[0, 40, 70, 100],
                                  labels=["LOW", "MED", "HIGH"]).astype(str)
        yield pdf[["cust_id", "score", "risk_band"]]   # Arrow batch out
custs = [(i, (i * 10) % 60, (i * 7) % 50, (i * 13) % 40) for i in range(1, 9)]
cust_df = spark.createDataFrame(custs, ["cust_id", "f1", "f2", "f3"]).repartition(3)  # 3 batches

scored = cust_df.mapInPandas(score_batches, "cust_id long, score double, risk_band string")
assert scored.count() == 8 and scored.filter("score IS NULL").count() == 0
assert {r["risk_band"] for r in scored.collect()} <= {"LOW", "MED", "HIGH"}
scored.orderBy("cust_id").show()

print("Talk track:")
print("  Why not toPandas(): the ENTIRE 10M-row frame is Arrow-serialized to the driver —")
print("  one JVM, one Python process, one heap. It didn't just risk OOM, it threw away all parallelism.")
print("  Why not row-UDF: pickling + Python-object per ROW + GIL = the classic 100x slowdown.")
print("  mapInPandas: Spark hands each task Arrow RECORD BATCHES; pandas vectorized ops run")
print("  inside the worker; results stream back as batches. All cores work, driver untouched.")
print("  Contract: (1) fixed in/out schema — declare the output schema string; (2) pure/deterministic —")
print("  the function may be re-executed on task retry, so NO side effects like external API calls")
print("  without idempotency; (3) any number of batches per partition, any order.")
print("  applyInPandas vs mapInPandas: applyInPandas gets ONE pandas frame PER GROUP (with the")
print("  grouping key contract) for group-wise models; mapInPandas is chunk-agnostic — right")
print("  for a row-wise scorer like this.")
print("  Memory bound: batch size ~ spark.sql.execution.arrow.maxRecordsPerBatch (default 10k)")
print("  — if a batch still OOMs, lower it or pre-filter wide columns before the map.")

## Q7 [ADVANCED] Spark Optimization — The Three Caching Layers: Disk Cache vs Spark Cache vs Result Cache

**Scenario**: A BI dashboard re-reads the same three gold Delta tables every 5 minutes. A well-meaning engineer added `df.cache()` to every query and the cluster's **storage memory is now exhausted**, evicting shuffle buffers and slowing everything down. Your lead asks: "Didn't Databricks cache things automatically anyway?"

**Requirements** — disentangle the layers:
1. **Disk cache** (a.k.a. Delta/IO cache): what it caches, format (vectorized?), when it's populated automatically, and what invalidates it.
2. **Spark cache** (`df.cache()` / `persist`): storage levels, when it is appropriate, and why it is NOT what a read-heavy BI workload needs.
3. **Result cache** (serverless SQL): what it dedupes, why it makes the first two mostly unnecessary for dashboards.
4. Give the decision rule for "cache or not" for: (a) a streaming foreachBatch sink, (b) an iterative ML loop, (c) the BI gold tables.

**Expected output**: an explanation with the hit-order of the layers and the correct call for each of the three cases.

In [0]:
# ── Solution Q7: the caching layers, in hit order ───────────────────────────
print("Hit order for a SELECT on a Delta table (top wins):")
print("  1. RESULT cache  (serverless SQL): exact-duplicate statements return instantly.")
print("  2. DISK cache    (aka IO/Delta cache): raw Parquet file blocks on LOCAL SSD in")
print("     VECTORIZED columnar format — automatic on reads, invalidated by file")
print("     rewrites (OPTIMIZE, vacuum, new writes) because it is keyed by file id.")
print("  3. SPARK cache   (cache()/persist): deserialized rows/batches in the JVM heap or")
print("     MEMORY_AND_DISK — explicit, manual, competes with execution memory.")
print()
print("Why the engineer's df.cache() hurt:")
print("  Spark cache lives in the SAME unified memory pool as shuffles/aggregations")
print("  (spark.memory.fraction, default 0.6). Caching three gold tables evicts execution")
print("  buffers -> spills -> the exact slowdown they tried to fix. Cached rows are also")
print("  invalidation-blind: Delta file changes silently make the cache stale in classic caches.")
print()
print("Decision rule per case:")
print("  (a) streaming foreachBatch sink   -> NO cache: batches are read once, cache is dead weight.")
print("  (b) iterative ML loop (N epochs)   -> YES persist(MEMORY_AND_DISK): genuine reuse 10x+;")
print("      unpersist() immediately after training — stale cache bugs are classic.")
print("  (c) BI gold tables                 -> rely on disk cache + result cache: automatic,")
print("      columnar, self-invalidating. Reach for persist only after a measured win.")
print()
print("  Interview one-liner: 'The best cache is the one the platform invalidates for you —")
print("  automatic beats explicit unless the reuse is measured.'")
print()
print("  Storage-level menu for persist(): MEMORY_ONLY (fast, evicts), MEMORY_AND_DISK")
print("  (safe default), DISK_ONLY (huge data, slow), MEMORY_AND_DISK_SER (compact bytes),")
print("  and on Databricks, DISK_ONLY is often fine because local NVMe is fast.")

## Q8 [ADVANCED] Spark Optimization — Diagnosing Shuffle Spill: Memory Fractions & Partition Sizing

**Scenario**: A 3 TB join in the Spark UI shows **"Spilled (disk): 800 GB"** in task metrics and takes 2.5 hours. The candidate fix on the table: "increase executors". Your job: diagnose properly.

**Requirements**:
1. Explain **unified memory management**: `spark.memory.fraction` (default 0.6), `spark.memory.storageFraction` (0.5), and what execution vs storage memory hold during a shuffle.
2. What spill actually is — why sort-merge join *sorts* on both sides, and when sorters exceed their pages.
3. Where to SEE spill (Spark UI / query profile task metrics) and the four fixes in order of preference: reduce shuffle volume, rebalance partitions, right-size memory, scale out.
4. Say why AQE coalesce (`spark.sql.adaptive` / serverless `auto` partitions) reduces — but cannot eliminate — spill.

**Expected output**: a triage script (what you check first) + the config talk-track.

In [0]:
# ── Solution Q8: spill triage + memory talk track ─────────────────────────
# Config probes (serverless shows 'auto' for AQE-managed shuffle partitions)
for c in ["spark.sql.shuffle.partitions", "spark.sql.files.maxPartitionBytes"]:
    print(f"{c} = {spark.conf.get(c)}")

print("\nUnified memory in one breath:")
print("  Heap is sliced: (1 - spark.memory.fraction) reserved for Spark/user objects;")
print("  the 0.6 fraction is the unified pool, soft-split at storageFraction (0.5) into")
print("  STORAGE (cached blocks) and EXECUTION (shuffle buffers, join sorts, agg hashes).")
print("  Execution can borrow idle storage — and when BOTH are full: SPILL.")
print()
print("What spilled 800 GB means here:")
print("  Sort-merge join sorts BOTH sides by key. Each partition's sorter must hold its")
print("  slice in memory pages; when a slice exceeds its execution share, UnsafeSorter")
print("  spills pages to disk. 800 GB spilled = partitions far too fat for their memory slice.")
print()
print("Triage order (memorize this — it is the interview):")
print("  1. REDUCE THE SHUFFLE: can the join be a broadcast join? (dimension ≤ a few GB)")
print("     Filter and select only needed columns BEFORE the join — shuffling 40 columns")
print("     to join on 2 is the #1 self-inflicted wound.")
print("  2. REBALANCE PARTITIONS: more shuffle partitions (or AQE skew-split) -> smaller")
print("     per-task slices. AQE coalesce fixes TOO-MANY-small-partitions, skew split fixes")
print("     one-fat-partition — but a uniformly fat shuffle still spills.")
print("  3. RIGHT-SIZE MEMORY: bigger executors (more heap in the unified pool) or fewer")
print("     concurrent tasks per executor (spark.executor.cores down) = more heap per task.")
print("  4. SCALE OUT only after 1-3 — 'add executors' with the same fat partitions just")
print("     buys more disks to spill onto.")
print()
print("Read the evidence: Spark UI -> stage -> task metrics: 'Spill (memory)' = pages")
print("evicted to heap-adjacent buffers, 'Spill (disk)' = written to local disk; both slow")
print("you ~5-10x vs pure in-memory shuffles. Query Profile shows it per operator.")

## Q9 [ADVANCED] ETL — Retroactive Corrections: Bitemporal Fact Tables

**Scenario**: The payments provider **corrects last week's transactions** — order 101 changes from $50 to $45, order 103 from $30 to $35 — with the corrections arriving days after the original events. Finance needs BOTH: "what did we report last Tuesday" (for the filed report) and "current truth" (for payouts). Auditors ask: *prove* which value was known on any date.

**Requirements**:
1. Design the fact table with **two time axes**: `effective_dt` (business/event time) and `ingested_dt` (when the row reached us). Explain why the classic "update in place" SCD1 merge fails auditors.
2. Write the **as-known-at** query: latest value per order among rows ingested ≤ a cutoff date.
3. Write the **restatement detector**: orders whose latest amount differs from the previously ingested amount (LAG over ingestion time).
4. State the retention/compaction policy so the table doesn't grow unbounded.

**Sample data & expected**: as-known-at 2026-01-09 → 101:$45, 102:$80, 103:$30 (the $35 correction arrived 01-10).

In [0]:
# ── Solution Q9: bitemporal orders — both time axes preserved ────────────────
orders_rows = [
    (101, "2026-01-05", "2026-01-06", 50.0),
    (101, "2026-01-05", "2026-01-08", 45.0),   # retroactive correction
    (102, "2026-01-06", "2026-01-07", 80.0),
    (103, "2026-01-07", "2026-01-09", 30.0),
    (103, "2026-01-07", "2026-01-10", 35.0),   # another late correction
]
bitemp = (spark.createDataFrame(orders_rows, ["order_id", "effective_dt", "ingested_dt", "amount"])
          .withColumn("effective_dt", F.to_date("effective_dt"))
          .withColumn("ingested_dt", F.to_date("ingested_dt")))
bitemp.createOrReplaceTempView("orders_bitemporal")

as_known_at = spark.sql("""
WITH ranked AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY ingested_dt DESC) AS rn
  FROM orders_bitemporal
  WHERE ingested_dt <= DATE'2026-01-09'          -- the 'as known at' cutoff
)
SELECT order_id, amount, ingested_dt FROM ranked WHERE rn = 1 ORDER BY order_id
""")
got = {r["order_id"]: r["amount"] for r in as_known_at.collect()}
assert got == {101: 45.0, 102: 80.0, 103: 30.0}, got   # 103's correction not yet known on 01-09
as_known_at.show()

restatements = spark.sql("""
SELECT order_id, effective_dt, ingested_dt, amount, prev_amount,
       amount - prev_amount AS delta
FROM (
  SELECT *, LAG(amount) OVER (PARTITION BY order_id ORDER BY ingested_dt) AS prev_amount
  FROM orders_bitemporal
)
WHERE prev_amount IS NOT NULL AND prev_amount != amount
ORDER BY order_id, ingested_dt
""")
assert restatements.count() == 2
restatements.show()

print("Talk track:")
print("  Why SCD1-in-place fails: overwriting $50 with $45 destroys the value finance")
print("  FILED last Tuesday — auditors cannot reconstruct what was known, when.")
print("  effective_dt answers 'when did the business event happen'; ingested_dt answers")
print("  'when did we learn it'. Every audit question is a WHERE on ingested_dt.")
print("  Retention: keep full history only for a window (e.g. 13 months); then compact")
print("  by keeping only the latest row per (order_id, effective_dt) older than the window.")
print("  Interview closer: 'bitemporal = SCD2 for FACTS, with the load date as the version axis.'")

## Q10 [ADVANCED] ETL — Data Contracts at the Producer Boundary

**Scenario**: The upstream `silver.orders` table changes type of `order_id` from BIGINT to STRING, adds a `promo_code` column, and once went 6 hours without data — each without warning. Consumers broke for two days. Leadership asks for "data contracts". Implement one.

**Requirements**:
1. Define a **versioned contract**: column name + type + nullability for every field, plus SLAs (freshness ≤ 60 min, min daily rows).
2. Build a validator that classifies drift as **BREAKING** (missing column, type change, nullability widened) vs **ADDITIVE** (new optional column) — with the versioning/evolution policy for each.
3. Define the **enforcement point**: fail the producer build vs quarantine vs alert — and who owns the contract.
4. Show the validator catching the drift below (expected: BREAKING type change on `order_id` + ADDITIVE `promo_code`).

**Expected output**: contract dict + validator + two demo DataFrames (clean, drifted) with classified reports.

In [0]:
# ── Solution Q10: schema contract validator ───────────────────────────────
def _norm(t):
    """Normalize type names across Spark versions (bigint/long etc.)."""
    m = {"bigint": "long", "integer": "int", "string": "string", "double": "double",
         "boolean": "boolean", "timestamp": "timestamp", "date": "date"}
    return m.get(t, t)

def validate_contract(df, contract):
    """Classify schema drift vs a versioned contract. Returns (report, is_ok)."""
    expected = {name: (_norm(dtype), nullable) for name, dtype, nullable in contract["columns"]}
    actual = {f.name: (_norm(f.dataType.simpleString()), f.nullable) for f in df.schema.fields}
    report = []
    for name, (dtype, nullable) in expected.items():
        if name not in actual:
            report.append(("BREAKING", f"missing column {name}"))
        else:
            a_dtype, a_null = actual[name]
            if a_dtype != dtype:
                report.append(("BREAKING", f"{name}: type {a_dtype} != contract {dtype}"))
            elif a_null and not nullable:
                report.append(("BREAKING", f"{name}: nullability widened (contract says NOT NULL)"))
    for name in actual:
        if name not in expected:
            report.append(("ADDITIVE", f"new column {name} (safe if downstream ignores it)"))
    return sorted(report), len(report) == 0

CONTRACT = {
    "table": "silver.orders",
    "version": "2.1.0",
    "owner": "orders-platform@company",
    "freshness_minutes": 60,          # SLA, checked separately at serve time
    "min_rows_per_day": 100_000,
    "columns": [("order_id", "bigint", False), ("customer_id", "bigint", False),
                ("amount", "double", True), ("status", "string", True)],
}

# clean batch — matches the contract exactly
clean = spark.createDataFrame([(1, 10, 99.5, "NEW")],
    "order_id BIGINT NOT NULL, customer_id BIGINT NOT NULL, amount DOUBLE, status STRING")
rep, ok = validate_contract(clean, CONTRACT)
assert ok and rep == [], rep

# drifted batch — order_id became STRING, promo_code appeared
drifted = spark.createDataFrame([("1", 10, 99.5, "NEW", "WINTER24")],
    "order_id STRING NOT NULL, customer_id BIGINT NOT NULL, amount DOUBLE, status STRING, promo_code STRING")
rep2, ok2 = validate_contract(drifted, CONTRACT)
assert not ok2 and {k for k, _ in rep2} == {"BREAKING", "ADDITIVE"}, rep2
print("Drift report:", rep2, sep="\n  ")

print("\nTalk track — enforcement & ownership:")
print("  Enforcement point: PRODUCER build (fail the PR), not the consumer 3am job.")
print("    BREAKING change -> contract must bump MAJOR version with a migration plan;")
print("    ADDITIVE -> MINOR bump, consumers stay safe because they never SELECT *.")
print("  Ownership: contract lives WITH the producer (PR-reviewed artifact); the platform")
print("    only renders it as checks (SDP expectations / CI schema diff) and alerts.")
print("  SLA side (freshness/volume) is monitored by the platform with SQL alerts —")
print("    see Q16/Q17: contracts stop problems at BUILD time; alerts catch the rest at RUN time.")

## Q11 [HARD] ETL — Building a RAG Ingestion Pipeline: Chunk → Embed → Vector Index *(2025-2026 trending)*

**Scenario**: The company is building a chatbot that answers from internal documents (policy PDFs, wiki pages, tickets). You own the **data pipeline** feeding it: 50k documents, growing daily, chunked, embedded, and served through a vector index. It must be **incremental and idempotent** — re-running a day must not duplicate chunks or re-embed unchanged content (embeddings cost real money).

**Requirements**:
1. Design the pipeline: source metadata → change detection → **chunking** (size + overlap) → content-hash dedup → embedding → vector index sync. State where each step can be safely re-run.
2. Implement: a chunker with overlap, a content-hash key that makes the chunks table idempotent (MERGE), and an incremental selection rule (only docs changed since the last watermark).
3. Explain the trade-offs: chunk size (too big / too small), overlap %, and what happens to the index when you CHANGE the chunking strategy (re-embed everything?).
4. Name the Databricks-native building blocks for each stage.

**Expected output**: runnable demo (chunk → hash → idempotent MERGE → watermark filter) + the design talk-track.

In [0]:
# ── Solution Q11: chunk → hash → idempotent MERGE + incremental selection ───
import hashlib

def chunk_text(text, size=12, overlap=3):
    """Word-window chunking with overlap (real pipelines chunk by TOKENS, ~256-512, 10-20% overlap)."""
    words = text.split()
    step = max(size - overlap, 1)
    chunks = []
    for i in range(0, len(words), step):
        piece = " ".join(words[i:i + size])
        if piece:
            chunks.append(piece)
        if i + size >= len(words):
            break
    return chunks

docs = [
    ("doc1", "2026-10-26 08:00", "delta lake provides acid transactions on object storage and scalable metadata handling"),
    ("doc2", "2026-10-26 08:00", "structured streaming uses micro batch execution with checkpoints for fault tolerance"),
    ("doc3", "2026-10-27 08:00", "unity catalog governs tables volumes and models with fine grained access control"),
]

rows = [(doc_id, ts, c, hashlib.md5(c.encode()).hexdigest())
        for doc_id, ts, text in docs for c in chunk_text(text)]
chunk_df = (spark.createDataFrame(rows, ["doc_id", "doc_ts", "chunk_text", "content_hash"])
            .withColumn("doc_ts", F.to_timestamp("doc_ts")))

# idempotent sink: MERGE on content_hash — replaying the same day changes nothing
TARGET = "workspace.default.d15_q11_chunks"
spark.sql(f"DROP TABLE IF EXISTS {TARGET}")
chunk_df.write.format("delta").saveAsTable(TARGET)
chunk_df.createOrReplaceTempView("new_chunks")
spark.sql(f"""
MERGE INTO {TARGET} AS t
USING new_chunks AS s
  ON t.content_hash = s.content_hash
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
n = spark.sql(f"SELECT COUNT(*) c, COUNT(DISTINCT content_hash) d FROM {TARGET}").first()
assert n["c"] == n["d"] == len(rows), "replay must not duplicate rows"
print(f"Idempotent sink verified: {n['c']} rows = {n['d']} distinct hashes even after a full replay")

# incremental selection: only docs changed after the last successful watermark
watermark = "2026-10-26 12:00"
changed_docs = chunk_df.filter(F.col("doc_ts") > F.lit(watermark).cast("timestamp"))
assert changed_docs.select("doc_id").distinct().count() == 1
print(f"Incremental pass would reprocess only {changed_docs.select('doc_id').distinct().count()} doc (doc3), not all 3")

print("\nTalk track — the production design:")
print("  Stage 1  source metadata table (doc_id, updated_at, raw_path) — the driver for change detection.")
print("  Stage 2  chunking: token-based 256-512 with 10-20% overlap so answers are not")
print("           cut mid-sentence; chunk strategy version is stamped on every chunk row.")
print("  Stage 3  content_hash = hash(chunk_text + chunk_version): dedups AND scopes")
print("           chunks to a strategy version — changing the strategy = new hashes =")
print("           controlled re-embed, old versions deleted after backfill (cost plan!).")
print("  Stage 4  embedding: batch job (Databricks Model Serving / ai functions / a hosted")
print("           embedding endpoint) reads only chunks missing vectors -> writes vectors column.")
print("  Stage 5  vector index: managed Vector Search index with SYNC (auto-refresh from the")
print("           Delta chunks table) or manual trigger after the embedding job finishes.")
print("  Re-run safety: every stage is keyed (doc_id -> content_hash -> vector id), so the")
print("  whole pipeline is idempotent — the exact property that makes reruns free of duplicates.")
print("  Interview bonus: mention evaluation datasets for retrieval quality + a re-embed")
print("  cost model before changing chunking — money is the thing interviewers probe.")

## Q12 [ADVANCED] Data Modeling — Surrogate Keys at Scale: Identity vs Hash vs Composite

**Scenario**: `dim_customer` has 500M rows and is **fully rebuilt every night** from source. The team currently joins facts on the 20-char business key and complains about slow joins; a previous attempt used `monotonically_increasing_id()` and orphaned half the fact table after a rebuild.

**Requirements**:
1. Compare **identity/sequence**, **hash (MD5/SHA)**, and **composite natural keys** for surrogate keys on: determinism across rebuilds, join cost, collision risk, and grain.
2. Explain the rebuild disaster with `monotonically_increasing_id()` — why it orphaned facts.
3. Implement hash surrogate keys and **prove** determinism across two independent rebuilds.
4. State the one rule that prevents stale surrogate keys in facts regardless of the strategy chosen.

**Expected output**: two rebuilt dims producing identical keys + the talk-track.

In [0]:
# ── Solution Q12: hash surrogate keys, proven deterministic across rebuilds ──

def build_dim():
    """Simulates the nightly full rebuild from source — same business keys each time."""
    rows = [(1, "acme", "VIP"), (2, "beta", "STD"), (3, "gamma", "STD")]
    return spark.createDataFrame(rows, ["bk_id", "name", "tier"])

dim_v1 = build_dim().withColumn("sk", F.md5(F.concat("bk_id", F.lit("|"), "name")))
dim_v2 = build_dim().withColumn("sk", F.md5(F.concat("bk_id", F.lit("|"), "name")))  # next night's rebuild

j = (dim_v1.select("bk_id", "sk")
         .join(dim_v2.select(F.col("bk_id").alias("bk_id2"), "sk"), "sk"))
assert j.count() == 3 and j.filter("bk_id != bk_id2").count() == 0
dim_v1.show(truncate=False)
print("Deterministic across rebuilds: every business key maps to the SAME sk in both builds")

print("\nTalk track — strategy comparison:")
print("  IDENTITY / SEQUENCE: cheap 8-byte joins, but keys are only stable if the table is")
print("    maintained incrementally (MERGE). A full rebuild reassigns numbers -> orphans.")
print("  monotonically_increasing_id(): partition-position based — NOT deterministic across")
print("    runs/repartitioning. Great for one-shot loads, a landmine for rebuilt dims.")
print("  HASH (md5/sha of business key): deterministic BY CONSTRUCTION -> full nightly rebuild")
print("    is safe; joins cost more than bigint but far less than a 20-char business key;")
print("    128-bit collision odds are astronomically low (universe-scale numbers).")
print("  COMPOSITE natural keys: no surrogate at all — simplest, but multi-column joins and")
print("    grain-creep bugs (a second natural key appears -> 'uniqueness' quietly breaks).")
print()
print("  The golden rule that prevents stale facts: facts store the surrogate resolved AT LOAD")
print("    TIME from the CURRENT dim — and on dim rebuilds you MERGE, never DELETE + reINSERT,")
print("    so historical fact keys always find their row (is_current = false rows stay).")

## Q13 [ADVANCED] Data Modeling — Modeling Ragged Hierarchies: Flatten vs Closure Table

**Scenario**: Your company org chart is 7 levels deep in some branches, 2 in others (a *ragged* hierarchy). Finance wants "revenue rollup by any manager's subtree" and the BI tool wants a clean drill-down.

**Requirements**:
1. Compare three storage options: **parent-child + recursive query at read time**, **flatten (L1..L7 columns)**, **closure/bridge table** (ancestor-descendant-depth). Give when-each-wins.
2. Build the closure table for the sample org below — every node paired with **all** its ancestors plus depth.
3. Explain what breaks when the hierarchy **changes** (a re-org), and how to keep SCD history of the tree.

**Sample org** (emp_id, manager_id): CEO(1) → VP-Eng(2), VP-Sales(3); 2 → TL-Data(4), TL-Apps(5); 3 → AE(6). Expected closure: 14 rows including self-pairs; depth(4→1) = 2.

In [0]:
# ── Solution Q13: closure table via iterative expansion to fixpoint ──────────
pc = spark.createDataFrame(
    [(1, None, "CEO"), (2, 1, "VP Eng"), (3, 1, "VP Sales"),
     (4, 2, "TL Data"), (5, 2, "TL Apps"), (6, 3, "AE")],
    ["emp_id", "manager_id", "name"])

# edge list: only NON-NULL parents (NULL manager = root — must not propagate!)
parent_of = (pc.filter(F.col("manager_id").isNotNull())
               .select(F.col("emp_id").alias("anc_id"), F.col("manager_id").alias("next_up")))

# seed: every node is its own ancestor at depth 0
closure_df = pc.select(F.col("emp_id").alias("descendant"),
                       F.col("emp_id").alias("ancestor"),
                       F.lit(0).alias("depth"))

prev, iterations = closure_df.count(), 0
while iterations < 6:                      # bounded by max hierarchy depth
    closure_df = closure_df.unionByName(
        closure_df.join(parent_of, closure_df["ancestor"] == parent_of["anc_id"])
                  .select("descendant", F.col("next_up").alias("ancestor"),
                          (F.col("depth") + 1).alias("depth"))
    ).dropDuplicates(["descendant", "ancestor"])
    n = closure_df.count()
    iterations += 1
    if n == prev:                          # fixpoint reached — no new pairs
        break
    prev = n

assert closure_df.count() == 14
assert closure_df.filter("descendant = 4 AND ancestor = 1").first()["depth"] == 2
closure_df.orderBy("descendant", "depth").show()

print("Talk track — the three options:")
print("  PARENT-CHILD + recursive CTE at read time: zero storage, but EVERY rollup query")
print("    walks the tree -> fine for 100 rows, painful for 100k at BI concurrency.")
print("  FLATTEN (L1..L7): great for fixed-depth BI drill-down; breaks on ragged depth,")
print("    needs full rebuild when depth grows, wide sparse rows.")
print("  CLOSURE TABLE: one row per (descendant, ancestor, depth). Rollup under manager X")
print("    = one indexed filter, any depth, ragged-proof. Cost: O(nodes x avg-depth) rows +")
print("    maintenance on re-orgs.")
print("  Re-orgs: rebuild closure in a nightly job (idempotent full rebuild is cheap); for")
print("    history, SCD2 the closure table (ancestor set + effective_ts) so tree-as-of-Q1")
print("    queries remain possible — that's the version interviews love.")

## Q14 [ADVANCED] Snowflake — Query Acceleration Service (QAS)

**Scenario**: A BI dashboard runs two heavy analytical queries that each scan billions of rows. Sizing the warehouse up to Large costs 4x. Someone suggests enabling the **Query Acceleration Service**.

**Requirements** — explain like a Snowflake-certified engineer:
1. What QAS actually does (which parts of a query get offloaded, and to where).
2. How to enable it (statement + scale-factor knob) and **how it is billed** — what makes it cheaper than resizing for scan-heavy workloads.
3. Which queries benefit and which do NOT (three counter-cases).
4. How you verify it is working after a week (management view / query history columns).

**Expected output**: enable/disable SQL + a verification query + the decision rule.

In [0]:
# ── Solution Q14: talk track + SQL snippets (Snowflake — illustrative strings) ─
qas_enable = """
ALTER WAREHOUSE bi_dash SET
  ENABLE_QUERY_ACCELERATION = TRUE
  QUERY_ACCELERATION_MAX_SCALE_FACTOR = 8;   -- how far QAS may scale serverless compute
"""
qas_verify = """
SELECT query_id, warehouse_name, start_time,
       query_acceleration_bytes_scanned,      -- > 0 means QAS offloaded scan work
       total_elapsed_time / 1000 AS sec
FROM   snowflake.account_usage.query_history
WHERE  start_time >= DATEADD(day, -7, CURRENT_TIMESTAMP())
  AND  query_acceleration_bytes_scanned > 0
ORDER  BY query_acceleration_bytes_scanned DESC;
"""
print(qas_enable)
print(qas_verify)

print("Talk track — the 60-second version:")
print("  QAS transparently offloads the SCAN-HEAVY portions of a query (large scans,")
print("  selective filters, small aggregations, joins that reduce data) to SERVERLESS")
print("  compute next to cloud storage — billed per-second, only while used.")
print("  Resizing the warehouse up pays for ALL queries 24/7; QAS pays only for the")
print("  heavy ones. For 2 whales + 100 minnows, QAS usually wins.")
print()
print("  Does NOT help: (1) queries hitting the RESULT cache anyway, (2) compute-bound")
print("  work (huge shuffles/spill — that needs a bigger warehouse), (3) tiny queries")
print("  whose overhead dominates. QAS is about SCAN, not about CPU or memory.")
print()
print("  Verification loop: enable -> watch query_history for query_acceleration_*")
print("  columns > 0 -> confirm billable QAS credits < the credits a resize would cost.")
print("  Governance: pair with a resource monitor on serverless credits so a runaway")
print("  dashboard cannot silently burn the QAS budget.")

## Q15 [ADVANCED] Snowflake — Snowpark: Python in the Warehouse vs Pushing Compute Elsewhere

**Scenario**: A feature-engineering team writes pandas. They ask: "Can we just do this inside Snowflake with Snowpark instead of exporting to a Spark cluster?" 40M rows, heavy group-wise transforms, output feeds an ML model in Databricks.

**Requirements**:
1. Explain what Snowpark actually is: Python DataFrames that **compile to SQL pushed down** into the warehouse, plus Python UDFs/stored procedures running inside the virtual warehouse.
2. Give the decision rule: **data gravity / governance / egress cost** vs **warehouse credit cost / ML ecosystem / debuggability**.
3. Show a short Snowpark snippet (DataFrame ops) and explain where it executes (client? warehouse?).
4. State when you would push the opposite way (move data OUT to Spark) and what changes at the billing meter.

**Expected output**: snippet + a two-column trade-off table in words.

In [0]:
# ── Solution Q15: talk track + Snowpark snippet (illustrative, not executed) ─
snowpark_snippet = """
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, avg, udf

session = Session.builder.configs(CONNECTION).create()

# LAZY: builds a query plan — nothing runs yet
df = (session.table("ANALYTICS.ORDERS")
        .filter(col("amount") > 0)                 # -> SQL WHERE, pushed to warehouse
        .group_by(col("customer_id"))
        .agg(avg("amount").alias("avg_order")))    # -> SQL GROUP BY

df.write.mode("overwrite").save_as_table("ANALYTICS.CUST_FEATURES")

# Python UDF: your python code runs INSIDE the warehouse compute
@udf(session=session, is_permanent=False)
def risk_band(avg: float) -> str:
    return "HIGH" if avg > 500 else "NORMAL"
"""
print(snowpark_snippet)

print("Talk track — where things execute:")
print("  DataFrame ops: translated to SQL, executed by the VIRTUAL WAREHOUSE (elastic,")
print("    multi-cluster Snowflake compute). The laptop driver only holds the PLAN.")
print("  Python UDFs / sprocs: shipped as staged code, executed on warehouse compute.")
print("  toPandas()/collect(): pulls data ACROSS the boundary — the expensive move.")
print()
print("Decision rule — keep compute IN the warehouse (Snowpark) when:")
print("  + data gravity: result is consumed inside Snowflake anyway")
print("  + governance: one permission boundary (roles, row policies) — no egress")
print("  + workload is SQL-shaped: filters, joins, group-bys — great push-down fit")
print("Move compute OUT to Spark/Databricks when:")
print("  + real ML training/inference (GPU, MLflow, spark.ml, feature stores)")
print("  + heavy custom Python that would burn warehouse credits running as UDFs")
print("  + you need the Spark ecosystem: streaming, Delta OPTIMIZE, SDP pipelines")
print()
print("Billing meter difference: Snowpark SQL work = warehouse credits (serverless")
print("credits for some sprocs); Spark = cluster or serverless compute. Same rule always:")
print("put compute where the DATA lives, unless the ecosystem forces the data to move.")

## Q16 [ADVANCED] Monitoring, Logging & Orchestration — Warehouse & Query Observability with `system.query.history`

**Scenario**: Your SQL warehouse bill spiked 40% this week. Nobody knows which queries or users drove it. Leadership wants a weekly report and a runaway-query alert.

**Requirements**:
1. Show the last queries executed on a warehouse with duration, bytes read, and executor — enough to find the culprit in one query.
2. Define a "runaway query" detection pattern (long-running OR huge scan) suitable for a SQL alert.
3. Explain the difference between **pipeline-level monitoring** (did the job run?) and this **query-level observability**, and how you would join both to cost (`system.billing.usage` via `warehouse_id`).

**Expected output**: a table of recent queries + a filter pattern that would have caught a 45-minute full-scan query.

In [0]:
# ── Solution Q16: query-level observability ──────────────────────────────────
# system.query.history is a UC system table — queryable from Spark/SQL just like any table.

qh = (spark.table("system.query.history")
          .select("start_time", "executed_by", "statement_type", "execution_status",
                  "total_duration_ms", "execution_duration_ms", "waiting_at_capacity_duration_ms",
                  "read_bytes", "read_rows", "produced_rows",
                  F.substring("statement_text", 1, 60).alias("statement"))
          .orderBy(F.col("start_time").desc())
          .limit(10))
qh.show(truncate=False)

# Runaway-query pattern: long-running OR huge scan (per Databricks' own examples)
runaway = spark.sql("""
SELECT statement_id, executed_by, total_duration_ms, read_bytes,
       ROUND(read_bytes / 1024.0 / 1024.0 / 1024.0, 1) AS read_gb,
       start_time
FROM system.query.history
WHERE start_time >= NOW() - INTERVAL 1 DAY
  AND (total_duration_ms > 15 * 60 * 1000 OR read_bytes > 500 * 1024 * 1024 * 1024)
ORDER BY read_bytes DESC
""")
print("Runaway candidates (last 24h):", runaway.count())

# Cost correlation: join query activity to billing by warehouse_id (pattern)
cost_pattern = """
SELECT u.usage_date, ROUND(SUM(u.usage_quantity * p.pricing.default), 2) AS usd
FROM   system.billing.usage u
LEFT JOIN system.billing.list_prices p
       ON u.sku_name = p.sku_name AND p.price_end_time IS NULL
WHERE  u.usage_metadata.warehouse_id = '<warehouse-id>'
GROUP BY u.usage_date ORDER BY u.usage_date DESC
"""
print("Talk track — the 3-layer observability stack:")
print("  1) PIPELINE level: did the job run, how long, did it fail? (Jobs UI, system.lakeflow.*)")
print("  2) QUERY level: which statements burned the warehouse? (system.query.history)")
print("  3) COST level: what did it cost? (system.billing.usage joined on warehouse_id)")
print("  Alert wiring: SQL alert over the runaway query above -> metric refresh every 15 min.")

## Q17 [ADVANCED] Monitoring, Logging & Orchestration — Alert Design That Doesn't Burn Out the On-Call Engineer

**Scenario**: Your data platform fires 200+ alerts a day. The on-call engineer mutes the channel. Last month a real outage was missed because it was alert #147 that day.

**Requirements**: Design the alerting pipeline end-to-end:
1. **Severity tiers** mapped to routing (P1 = page, P2 = team Slack, P3 = daily digest).
2. **Deduplication**: same (service, alert-code) within a 15-minute window collapses to ONE notification.
3. **Escalation**: what happens if a P1 is unacknowledged for 10 minutes?
4. **Actionability rule**: every page must link a runbook — explain why this rule is the one that actually reduces noise.

**Expected output**: a small simulation that reduces 5 raw events to a 4-entry routed digest.

In [0]:
# ── Solution Q17: alert dedup + severity routing simulation ───────────────────
import datetime as dt

SEVERITY = {
    "FRESHNESS_BREACH": ("P1", "page on-call + runbook link"),
    "ROW_COUNT_DROP":   ("P2", "slack #data-alerts"),
    "SCHEMA_DRIFT":     ("P2", "slack #data-alerts"),
    "COST_ANOMALY":     ("P3", "daily digest email"),
}

def build_digest(events, window_min=15):
    """Collapse duplicate (service, code) alerts inside a rolling window; route by severity."""
    last_seen, digest = {}, []
    for ts_str, service, code in sorted(events):   # events must arrive time-ordered
        ts = dt.datetime.strptime(ts_str, "%Y-%m-%d %H:%M")
        key = (service, code)
        if key in last_seen and (ts - last_seen[key]).total_seconds() / 60.0 <= window_min:
            continue                                  # duplicate inside the window -> swallow
        last_seen[key] = ts
        sev, route = SEVERITY[code]
        digest.append((ts_str, service, code, sev, route))
    return digest

raw_events = [
    ("2026-10-27 03:01", "payments_gold", "FRESHNESS_BREACH"),
    ("2026-10-27 03:04", "payments_gold", "FRESHNESS_BREACH"),  # dup: 3 min later
    ("2026-10-27 03:05", "payments_gold", "ROW_COUNT_DROP"),
    ("2026-10-27 03:20", "orders_silver", "SCHEMA_DRIFT"),
    ("2026-10-27 03:22", "payments_gold", "FRESHNESS_BREACH"),  # NEW window (>15 min)
]
digest = build_digest(raw_events)
assert len(digest) == 4 and digest[0][1:] == ("payments_gold", "FRESHNESS_BREACH", "P1", "page on-call + runbook link")
for entry in digest:
    print(entry)
print(f"{len(raw_events)} raw events -> {len(digest)} routed digest entries")

print("\nTalk track — the full design:")
print("  Severity by IMPACT, not by source: freshness on a serving table = P1;")
print("    a column rename nobody consumes yet = P3 digest.")
print("  Dedup key = (service, alert_code) + window; re-alert only if still firing or new window.")
print("  Escalation: P1 unacked 10 min -> secondary on-call; 20 min -> manager. PagerDuty does this natively.")
print("  Actionability rule: NO notification without a runbook URL. An alert you can't act on")
print("    is a report, not an alert — route it to the digest instead.")
print("  Review ritual: weekly, delete or downgrade any alert nobody acted on — alert debt is real debt.")

## Q18 [HARD] Monitoring, Logging & Orchestration — Data Quality Frameworks: Build vs Buy

**Scenario**: Your CDO asks: "Should we standardize on Great Expectations, dbt tests, Databricks Lakehouse Monitoring, or Soda? Or build our own?" You have 40 pipelines across 4 teams.

**Requirements** — answer like a staff engineer making a recommendation:
1. Compare the four options across: **where checks run, learning curve, integration with Delta/UC, alerting, OSS momentum, operating cost**.
2. Describe the **hybrid pattern** most real platforms converge on.
3. Name the checks that belong to a **contract** (owned by producers) vs **monitors** (owned by the platform) vs **orchestrator alerts** (freshness/volume).

**Expected output**: a decision matrix + a one-paragraph recommendation.

In [0]:
# ── Solution Q18: decision matrix (talk track) ────────────────────────────────
print("Option          Where checks run        Strengths                        Watch out for")
print("───────────────  ──────────────────────  ────────────────────────────────  ─────────────────────────────")
print("Great Expect.   Your Spark/Python code  Huge check library, data docs     YAML sprawl, python-packaging ops,")
print("                (batch)                 Pandas + Spark backends           team turnover knowledge loss")
print("dbt tests       In the warehouse (SQL)   Native to transform layer,        Only where dbt owns the model,")
print("                                         cheap to adopt for SQL teams     blind to non-dbt pipelines")
print("Lakehouse Mon.  Databricks-managed       Zero infra, drift + freshness     Bind to UC tables; less flexible")
print("                snapshot/timeseries      metrics + custom SQL metrics     than code-based suites")
print("Soda            CLI / orchestrator       Business-friendly rules language  Another tool in the stack; checks")
print("                agents                   + catalog integration             not near the transform")
print()
print("Hybrid that most platforms converge on:")
print("  1) PRODUCERS own CONTRACTS (schema + not-null/uniqueness) — fail the build, not the alert.")
print("  2) PLATFORM owns MONITORS (Lakehouse Monitoring: drift, freshness, custom metrics on gold tables).")
print("  3) ORCHESTRATOR owns SLA alerts (freshness/volume per pipeline) wired to the on-call digest (Q17).")
print()
print("Recommendation (staff-engineer version):")
print("  Start with what the platform gives you for free (Lakehouse Monitoring + SQL alerts).")
print("  Add dbt tests if dbt is already your transform layer. Introduce GE/Soda only when teams need")
print("  rich custom suites + data docs. Never build your own framework — you will rebuild all four badly.")
print()
print("The interview-winning line: 'Choose by WHERE failures should STOP, not by framework popularity —")
print("contracts stop at build time, monitors stop at serve time, alerts wake a human.'")

## 🏁 Day 15 Score Card

Track your performance — be honest, the point is to find gaps *before* the real interview.

| Q | Topic | Tag | Attempted (✓/✗) | Confident (1-5) | Revisit? |
| --- | --- | --- | --- | --- | --- |
| Q1 | Trailing-window anomaly detection (z-score band) | ADVANCED | | | |
| Q2 | Range joins: double-booked rooms | ADVANCED | | | |
| Q3 | Subarray sum K (prefix sums) | ADVANCED | | | |
| Q4 | Trie autocomplete engine | ADVANCED | | | |
| Q5 | Mutual friends (PySpark graph) | ADVANCED | | | |
| Q6 | Batch scoring with `mapInPandas` | ADVANCED | | | |
| Q7 | The three caching layers | ADVANCED | | | |
| Q8 | Diagnosing shuffle spill | ADVANCED | | | |
| Q9 | Bitemporal facts & retroactive corrections | ADVANCED | | | |
| Q10 | Data contracts at the boundary | ADVANCED | | | |
| Q11 | RAG ingestion pipeline (chunk → embed → vector) | HARD | | | |
| Q12 | Surrogate keys at scale | ADVANCED | | | |
| Q13 | Ragged hierarchies & closure tables | ADVANCED | | | |
| Q14 | Snowflake Query Acceleration Service | ADVANCED | | | |
| Q15 | Snowpark vs pushing compute elsewhere | ADVANCED | | | |
| Q16 | Query observability (`system.query.history`) | ADVANCED | | | |
| Q17 | Alert design that doesn't burn out on-call | ADVANCED | | | |
| Q18 | DQ frameworks: build vs buy | HARD | | | |

**Self-review prompts:**

1. Which question took the longest to articulate out loud? Practice it twice more tomorrow.
2. For every ✗: write the one-sentence core idea on a flashcard (e.g. *"z-score needs σ of the window WITHOUT today in it"*).
3. Week 3 has begun — interviews at product companies test **trade-off articulation**, not recall. For each question, say the trade-off out loud: *what would I give up if I chose the other design?*
4. Tomorrow: Day 16 (ADVANCED — deepening into HARD territory).